# Kaggle Housing Price Prediction - Linear Model

Ridge regression solution for the [Kaggle Housing Prices Competition for Kaggle Learn Users](https://www.kaggle.com/competitions/home-data-for-ml-course). Predicts residential home sale prices in Ames, Iowa. The competition metric is MAE on the raw price, but the model is trained on log(SalePrice) to reduce the influence of high-value outliers. This is equivalent to minimising RMSLE.

Ridge was selected because there are ~1460 rows of data and 80-90 features (Before one-hot encoding) in the model so we don't have to worry too much about droping features. There is also significant multicollinearity between many of the columns so Ridge should handle this better than something like LASSO that would totally drop some correlated features.

Exploratory analysis and outlier identification are covered in `data-exploration.ipynb`. This notebook focuses on the modelling pipeline:

ADD PIPELINE

TODO:
- Try Huber regression. This should deal with outliers better and might help with reducing sensitivity to strange houses

In [17]:
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import matplotlib.pyplot as plt

import scipy.stats as stats
from scipy.optimize import minimize_scalar

from pandas.api.types import CategoricalDtype

from sklearn.linear_model import Ridge

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict
from sklearn.feature_selection import mutual_info_regression

import joblib
import plotly
import optuna
import optuna.visualization as vis

# set global theme and disable frames
sns.set_theme(style = 'white',
              rc={'legend.frameon': False},
              )
sns.set_style('ticks', {'xtick.major.size' : 8,
                        'ytick.major.size' : 8,
                        'xtick.bottom'     : True,
                        'ytick.left'       : True,
                        })

## Preprocessing

In [3]:
def clean(df):
    '''
    Applies preliminary data cleaning to the provided
    dataframe based on the data exploration notebook
    '''
    # set missing GarageYrBlt values to YearBuilt
    df.loc[df['GarageYrBlt'] < df['YearBuilt'], 'GarageYrBlt'] = df['YearBuilt']

    # drop columns that are missing a significant amount of data
    df = df.drop(columns=['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType'])

    # note: FireplaceQu has >40% missing values but can be imputed with
    # ordinal encoding so might be worth keeping

    return df


def remove_outliers(df):
    '''
    Two Edwards neighbourhood partial sales identified in data-exploration.ipynb that
    seem like outliers. Large area, good quality, but low price. Remove these
    from the training data
    '''
    mask = ((df['Neighborhood'] == 'Edwards') &
            (df['SaleCondition'] == 'Partial') &
            (df['GrLivArea'] > 4000))
    
    return df[~mask]


def impute(df, impute_stats=None):
    '''
    Impute the columns in the provided dataframe
    based on the results of the cells above

    impute_stats is a dict that specifies values to
    use for imputing that need to be collected from
    training data 
    '''
    # first handle special cases

    # estimate missing LotFrontage by the root of LotArea
    df['RootArea']    = np.sqrt(df['LotArea'])
    df['LotFrontage'] = df['LotFrontage'].fillna(df['RootArea'])
    df = df.drop(columns=['RootArea'])

    # set missing GarageYrBlt with YearBuilt
    df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['YearBuilt'])

    # impute the missing Electrical data with the mode
    if impute_stats is None:
        impute_stats = {'Electrical' : df['Electrical'].mode()[0]}
    
    df['Electrical'] = df['Electrical'].fillna(impute_stats['Electrical'])


    # general handling for the rest of the columns
    for col in df.select_dtypes('object', 'category'):
        df[col] = df[col].fillna('NA')
    
    for col in df.select_dtypes('number'):
        df[col] = df[col].fillna(0)
    
    return df, impute_stats


def encode(df):
    '''
    Encode the categorical columns of the provided dataframe
    based on the encoding outlined below
    '''
    # categorical columns
    categorical_cols = set(df.select_dtypes('object', 'category').columns)

    nominative_cols = ['MSZoning', 'Street', 'LandContour', 'Utilities', 'LotConfig',
                       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
                       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'Foundation',
                       'Heating', 'Electrical', 'Functional', 'SaleType', 'SaleCondition',
                       'GarageType']

    ordinal_cols = list(categorical_cols - set(nominative_cols))

    ordinal_encoding = {'LotShape'      : ['Reg', 'IR1', 'IR2', 'IR3'],
                        'LandSlope'     : ['Gtl', 'Mod', 'Sev'],
                        'ExterQual'     : ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
                        'ExterCond'     : ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
                        'BsmtQual'      : ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
                        'BsmtCond'      : ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
                        'BsmtExposure'  : ['No', 'Mn', 'Av', 'Gd'],
                        'BsmtFinType1'  : ['Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
                        'BsmtFinType2'  : ['Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
                        'HeatingQC'     : ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
                        'CentralAir'    : ['N', 'Y'],
                        'KitchenQual'   : ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
                        'FireplaceQu'   : ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
                        'GarageQual'    : ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
                        'GarageCond'    : ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
                        'GarageFinish'  : ['Unf', 'RFn', 'Fin'],
                        'PavedDrive'    : ['N', 'P', 'Y'],
                        }

    ordinal_encoding = {key: ['NA'] + values for key, values in ordinal_encoding.items()}

    for col in nominative_cols:
        df[col] = df[col].astype('category')
    
    for col in ordinal_cols:
        df[col] = df[col].astype(CategoricalDtype(ordinal_encoding[col], ordered=True))
    
    return df

## Load Data

Apply the full preprocessing pipeline. The `impute_stats` dict is fit on the training set and passed through to the test set so the test data never influences any computed statistics.

In [4]:
def load_data():
    # Load the Kaggle dataset
    path_train = './input/train.csv.gz'
    path_test  = './input/test.csv.gz'

    train_full = pd.read_csv(path_train, compression='gzip', index_col='Id')
    test_full  = pd.read_csv(path_test, compression='gzip', index_col='Id')

    train = train_full.copy()
    train = clean(train)
    train = remove_outliers(train)
    train, impute_stats = impute(train)
    train = encode(train)

    test = test_full.copy()
    test = clean(test)
    test, _ = impute(test, impute_stats)
    test = encode(test)

    return train, test

train, test = load_data()

print(f'Shape of train dataset: {train.shape}')
print(f'Shape of test dataset:  {test.shape}')

# remove training rows with missing target (SalePrice)
print(f'{train["SalePrice"].isna().sum()} trainig rows have a missing SalePrice')
train = train.dropna(subset=['SalePrice']) # no missing SalePrice so not needed

# remove any duplicate training rows
print(f'{train.duplicated().sum()} trainig rows are duplicates')
train = train.drop_duplicates(keep='first') # no duplicates so not needed

Shape of train dataset: (1458, 75)
Shape of test dataset:  (1459, 74)
0 trainig rows have a missing SalePrice
0 trainig rows are duplicates


## Baseline Model

Establish a baseline score using Ridge with a default alpha parameter on the raw features before any feature engineering. The scoring function uses 5-fold cross-validation and reports RMSE on log(`SalePrice`), consistent with the competition's RMSLE metric.

In [11]:
def cat_to_codes(X):
    '''
    Convert ordinal CategoricalDtype columns to integer codes so that
    Ridge can understand the ranking
    '''
    X = X.copy()
    for col in X.select_dtypes('category'):
        if X[col].cat.ordered:
            X[col] = X[col].cat.codes

    return X


def prepare_features_linear(X):
    '''
    Convert categoricals for Ridge. Ordinal columns are converted to
    int codes, nominal columns are converted to one-hot encoding
    '''
    X = cat_to_codes(X)
    X = pd.get_dummies(X, drop_first=True)
    return X


def score_model(X, y):
    '''
    Return a score using cross validation to estimate
    model performance. This competition uses RMSE between
    log prices so use that as a metric
    '''
    ridge_pipeline = Pipeline([('scaler', StandardScaler()),
                               ('model',  Ridge(random_state=123)),
                               ])
    log_y    = np.log(y)
    X_linear = prepare_features_linear(X)

    scores = cross_val_score(ridge_pipeline, X_linear, log_y, cv=5, scoring='neg_mean_squared_error')
    score  = -1 * scores.mean()
    score  = np.sqrt(score)

    return score

In [14]:
X = train.copy()
y = X.pop('SalePrice')

baseline_score = score_model(X, y)
print(f'Baseline model score: {baseline_score:.5f} RMSE')

Baseline model score: 0.12270 RMSE


In [ ]:
# have to normalize data. Plot skew and number non-zero entries to see which columns would be best to normalize with a log